# Dynamic EGF network, version 2

Successor to `dynamic_network_optimization_tutorial.ipynb`, which is left untouched so the two
runs stay comparable. Every function lives in **`src/dynamic_network_v2.py`**; this notebook holds
parameters, calls and narrative only.

The full specification, including the reasoning behind every rule, is
**`Claude_promts/dynamic_network_instrucitons.md`**. Section numbers below refer to it.

## What is built so far — step 1 of §9: the site table

Four things change relative to version 1:

1. **Responsiveness replaces amplitude.** Version 1 kept the top 10% of sites by absolute peak
   fold change. That is an amplitude criterion on unnormalised data, so it partly ranks sites on
   the global loading shift. Here the limma omnibus *F* decides: did this site move more than its
   own replicate noise (§2.1)?
2. **The sign is kept.** Version 1 used `|log2FC|` throughout, so a dephosphorylation was
   rewarded exactly like a phosphorylation and was allowed to switch its kinase on (§2.3).
3. **Multi-site peptides are exploded.** Version 1 built its lookup key as
   `protein_Id + '_' + PhosSites`, which for a two-site peptide gives `Q00000_S416;S417` — a
   string that can never match a kinase-substrate target. Those sites were silently unmatchable
   (§2.4).
4. **The activation time is a parameter, not a hard-wired column.** `TIME_DEFINITION` dispatches,
   so the two alternative definitions in §3 can be added later without touching anything
   downstream.

Not yet built: the candidate graph, the ILP, the SIGNOR causal layer, the drawing (steps 2-6).

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

# Project code is imported as `from src.xxx import yyy`, so the repo root has to be on sys.path
# whether the notebook is run from its own folder or from the project root.
REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dynamic_network_v2 import (
    ACTIVATION_LOOP_PAIRS,
    DEFAULT_MAX_FFDR,
    STIMULATION_TIMEPOINTS,
    TIME_DEFINITIONS,
    activation_loop_report,
    build_site_table,
    site_table_summary,
    trim_site_table,
)

pd.set_option("display.width", 200)
print("Repo root:", REPO_ROOT)

Repo root: /Users/ignacionavascamacho/PycharmProjects/TMT_Data_analysis


## 1. Parameters

Everything the run depends on, in one cell.

| parameter | why it is set this way |
|---|---|
| `DATA_PATH` | fixed by the user on 2026-09-16 (§2.5). Note it **predates the 2026-09-15 channel-offset correction**, so the unnormalised-data caveat in `CLAUDE.md` → *Known issues* applies to every fold change used below |
| `MAX_FFDR` | limma omnibus *F* cutoff. `None` switches the gate off |
| `TIME_DEFINITION` | `"peak_fc"` — the simple start agreed on 2026-09-16. `"max_step"` and `"onset"` are specified in §3 and raise `NotImplementedError` |
| `MIN_REPS` | `None`: the FFDR gate already drops every site limma could not test, which is exactly the `n:reps == 1` set |

In [2]:
DATA_PATH = REPO_ROOT / "Experiment/hme1_2/Data/Processed/20260819_hTERT_HEM1_2_log2_processed_steps.tsv"
RESULTS_DIR = REPO_ROOT / "notebooks/07_dynamics_RPCST"

CELL_LINE = "WT"
CONDITION = "EGF"
MAX_FFDR = DEFAULT_MAX_FFDR   # 0.05
TIME_DEFINITION = "peak_fc"   # sections 1-13 keep peak_fc; section 14 measures why "onset" replaced it as the module default
MIN_REPS = None

OUTPUT_PREFIX = RESULTS_DIR / f"v2_site_table_{CONDITION}_ffdr{str(MAX_FFDR).replace('.', 'p')}_{TIME_DEFINITION}"

print("Data:", DATA_PATH.name)
print("Exists:", DATA_PATH.exists())
print("Time definitions available:", TIME_DEFINITIONS)
print("Output prefix:", OUTPUT_PREFIX.name)

Data: 20260819_hTERT_HEM1_2_log2_processed_steps.tsv
Exists: True
Time definitions available: ('peak_fc', 'max_step', 'onset')
Output prefix: v2_site_table_EGF_ffdr0p05_peak_fc


## 2. Confirm the table carries what the pipeline needs

The first action of step 1 (§2.5): the column list was never verified when the spec was written.
The FFDR gate depends on `WT_log2:FFDR_EGF_omnibus` existing — if it were absent, limma would have
to be merged in first, and the gate must never be skipped silently.

In [3]:
columns = pd.read_csv(DATA_PATH, sep="\t", nrows=0).columns.tolist()

required = (["protein_Id", "protein_name", "description", "PhosSites", "n:reps",
             "ReferenceIntensity", "ON_FUNCTION", "functional_score",
             f"{CELL_LINE}_peak:FC_{CONDITION}", f"{CELL_LINE}_log2:FFDR_{CONDITION}_omnibus"]
            + [f"{CELL_LINE}_log2:FC_{CONDITION}_{t}" for t in STIMULATION_TIMEPOINTS]
            + [f"{CELL_LINE}_log2:step_{CONDITION}_{t}" for t in STIMULATION_TIMEPOINTS])

missing = [column for column in required if column not in columns]
print(f"{len(columns)} columns in the file")
print("Missing required columns:", missing if missing else "none")
assert not missing, f"Required columns absent: {missing}"

501 columns in the file
Missing required columns: none


## 3. Build the site table

`build_site_table` runs the whole of step 1 and returns a **filter ledger** alongside the table,
so every row that leaves the dataset is accounted for. Read the ledger before the table.

Note the negative `dropped` on the explosion row: that step *adds* rows, one per residue of a
multi-phosphorylated peptide.

In [4]:
site_table, ledger = build_site_table(DATA_PATH,
                                      max_ffdr=MAX_FFDR,
                                      time_definition=TIME_DEFINITION,
                                      cell_line=CELL_LINE,
                                      condition=CONDITION,
                                      min_reps=MIN_REPS,)

print(ledger.to_string(index=False))

                                    step  rows  dropped
                                  loaded 49943      NaN
         contaminants and decoys removed 49943      0.0
             localized phosphosites only 34675  15268.0
           limma omnibus F, FFDR <= 0.05 15113  19562.0
         valid activation time (peak_fc) 15113      0.0
                peak fold change present 15113      0.0
multi-site peptides exploded to residues 17291  -2178.0
    collapsed to one row per phosphosite 14797   2494.0


In [5]:
print(site_table_summary(site_table).to_string(index=False))

                  metric     value
                   sites 14797.000
                proteins  4051.000
            up-regulated 11601.000
          down-regulated  3196.000
from multi-site peptides  2926.000
  measured on >1 peptide  1985.000
         median |log2FC|     0.420
            max |log2FC|     5.651
           |log2FC| >= 1  1602.000
           peak at 2 min   476.000
           peak at 5 min  8331.000
          peak at 10 min  2541.000
          peak at 15 min  2743.000
          peak at 90 min   706.000


In [6]:
site_table[["protein_site_id", "kinsub_site_id", "activation_time",
            "peak_log2_fc_vs_starve", "peak_direction", "n:reps",
            "n_sites_on_peptide", "n_source_rows"]].head(15)

,protein_site_id,kinsub_site_id,activation_time,peak_log2_fc_vs_starve,peak_direction,n:reps,n_sites_on_peptide,n_source_rows
0,EGFR_Y1197,P00533_Y1197,5,5.651342,up,3,2,2
1,EGFR_T1191,P00533_T1191,5,5.651342,up,3,2,1
2,HGS_Y289,O14964_Y289,90,5.045790,up,2,1,1
3,PML_S505,P29590_S505,10,4.960102,up,3,2,3
4,PML_S512,P29590_S512,10,4.960102,up,3,2,1
5,AHNAK_S440,Q09666_S440,10,4.668395,up,4,1,1
6,STAM2_Y192,O75886_Y192,10,4.603784,up,4,1,1
7,GAB1_T638,Q13480_T638,5,4.580855,up,2,2,1
8,GAB1_Y627,Q13480_Y627,5,4.580855,up,2,2,2
9,PLCG1_Y1253,P19174_Y1253,5,4.456438,up,4,1,1


### What the composition says

Two numbers to read sceptically, both consequences of the missing sample-loading normalisation
recorded in `CLAUDE.md` → *Known issues* (median `log2:FC` **+0.566** across all sites at EGF
5 min):

- the strong **up/down imbalance**, and
- the pile-up of peaks at **5 min**, the timepoint carrying the largest global shift.

Neither invalidates the run, but both say the prize and the activation time inherit a technical
offset. The sensitivity check is to re-run on a `chcorr` table once the pipeline works (§2.5).

## 4. Does the activation time survive its own test?

Both residues of a kinase activation loop are phosphorylated by the same event, so any sound time
definition must give them the **same** activation time. This is the objective test proposed in §3
for choosing between `peak_fc`, `max_step` and `onset` — run here against `peak_fc` to see what
the simple choice costs.

Version 1 failed it badly: ERK1 Y204 at 5 min against T202 at **90 min**, which then dragged
ERK1's latent activation time to 90 — biologically wrong, ERK1 activates within 2-5 min.

In [7]:
loops = activation_loop_report(site_table, ACTIVATION_LOOP_PAIRS)
print(loops.to_string(index=False))

detected = loops["both_detected"].sum()
agreeing = loops["times_agree"].sum()
print(f"\n{agreeing} of {detected} detected pairs agree, out of {len(loops)} pairs tested")

protein_Id                      description      site_a      site_b time_a log2fc_a time_b log2fc_b  both_detected  times_agree
    P27361         ERK1 TEY activation loop P27361_T202 P27361_Y204     10    2.897      5    3.428           True        False
    P28482         ERK2 TEY activation loop P28482_T185 P28482_Y187     10    3.214     10    3.214           True         True
    Q02750             MEK1 activation loop Q02750_S218 Q02750_S222   <NA>     <NA>   <NA>     <NA>          False        False
    P36507             MEK2 activation loop P36507_S222 P36507_S226   <NA>     <NA>   <NA>     <NA>          False        False
    P31749 AKT1 activation, PDK1 and mTORC2 P31749_T308 P31749_S473   <NA>     <NA>   <NA>     <NA>          False        False

1 of 2 detected pairs agree, out of 5 pairs tested


## 5. Save

Processed tables are written as `.tsv`, and an existing file is never overwritten — a run with
different parameters gets a different name through `OUTPUT_PREFIX`.

In [8]:
# The source table carries ~500 columns; trim_site_table keeps the identifiers, the response,
# the annotations the sign rule and the drawing need, and this condition's profiles (§8).
trimmed = trim_site_table(site_table,
                          cell_line=CELL_LINE,
                          condition=CONDITION,)
print("Columns kept:", trimmed.shape[1], "of", site_table.shape[1])

output_path = Path(str(OUTPUT_PREFIX) + ".tsv")
if output_path.exists():
    print("Exists already, not overwritten:", output_path.name)
else:
    trimmed.to_csv(output_path, sep="\t", index=False)
    print("Written:", output_path.name, trimmed.shape)

ledger_path = Path(str(OUTPUT_PREFIX) + ".ledger.tsv")
if not ledger_path.exists():
    ledger.to_csv(ledger_path, sep="\t", index=False)
    print("Written:", ledger_path.name)


Columns kept: 43 of 511
Exists already, not overwritten: v2_site_table_EGF_ffdr0p05_peak_fc.tsv


## 6. Observations, 2026-09-16

**The multi-site fix repaired the ERK1 timing.** In the raw table, the T202-only peptide peaks at
90 min and the Y204-only peptide at 5 min — the disagreement version 1 inherited. The
**doubly**-phosphorylated `T202;Y204` peptide peaks at 10 min and carries a larger fold change, so
once multi-site peptides are exploded and the strongest peptide per site wins, T202 is re-assigned
from 90 min to 10 min. The pair still disagrees by one grid step (10 vs 5), but the 85-minute error
is gone. ERK2 T185/Y187 agree exactly, at 10 min.

**Three activation loops are not measured at all** in this dataset — MEK1 S218/S222, MEK2
S222/S226 and AKT1 T308/S473 are absent from the table, not filtered out by the gate. Consequences
for step 2: MEK carries only regulatory/feedback sites here (S298, T292, Y300, S299), so **MEK
activation cannot be read directly** and the EGFR → ERK route has to be reconstructed from its
surroundings rather than measured at the node itself. This is an argument for the SIGNOR causal
layer in §4.3, not against it.

**KRAS has no rows at all**, as expected for a protein whose activation is nucleotide exchange
rather than phosphorylation. It can only ever enter the network as an unmeasured Steiner node via
SIGNOR — which is exactly the gap that makes EGFR → GRB2/SOS1 → RAS → RAF unrepresentable today.


# Step 2 — the candidate graph and the ILP

Everything here is §4 and §5 of the spec. Three changes relative to version 1:

1. **Edges are cut, not sites** (§4.0). The kinase-substrate table is dominated by lenient motif
   predictions, so without filtering each site carries ~43 candidate kinases and the ILP has
   ~570 000 binary edge variables. `RESOURCE_REGEX` and `MAX_KINASES_PER_SITE` bring that to ~26 000
   edges while keeping every site.
2. **Any site may be a leaf** (§4.1). Version 1 required every selected node to have a selected
   outgoing edge, so every path had to end at a transcription factor; sites that are not kinases or
   TFs were structurally unselectable and ~98% of the prize was a constant. A path may now stop
   anywhere, which is also what makes feedback expressible.
3. **Connectivity by single-commodity flow** instead of MTZ distance labels (§5) — same rooted
   guarantee, tighter relaxation.

In [9]:
import collections
import shutil
import subprocess

import pickle
import time

import networkx as nx

from src.dynamic_network_v2 import (
    DEFAULT_RESOURCE_REGEX,
    EGFR_UNIPROT,
    add_orphan_root_edges,
    add_propagation_rule,
    build_candidate_graph,
    load_kinase_substrate_edges,
    orphan_report,
    select_network,
    selection_tables,
    tag_feedback_edges,
    validate_selection,
)

KINSUB_PKL = REPO_ROOT / "External_Data/Metadata/Martin/combined_kinsub_ce6e091017a229841db87586decbe46e.pkl"

RESOURCE_REGEX = DEFAULT_RESOURCE_REGEX   # "Strict|literature"
MAX_KINASES_PER_SITE = None               # see section 8.1: the rank cap severs the root's own edges
NODE_COST = 0.3                           # re-tuned in section 9.1; 1.2 was inherited from version 1
MIP_GAP = 0.02
TIME_LIMIT_S = 900

print("Kinase-substrate table:", KINSUB_PKL.name)
print("Resource filter:", RESOURCE_REGEX, "| max kinases per site:", MAX_KINASES_PER_SITE)

Kinase-substrate table: combined_kinsub_ce6e091017a229841db87586decbe46e.pkl
Resource filter: Strict|literature | max kinases per site: None


## 7. Kinase-substrate edges

`load_kinase_substrate_edges` reports what each stage of the filtering costs. Read `edges_per_site`:
it is the number the graph size really depends on.

In [10]:
edges, edge_stats = load_kinase_substrate_edges(KINSUB_PKL,
                                               site_ids=set(site_table["kinsub_site_id"]),
                                               resource_regex=RESOURCE_REGEX,
                                               max_kinases_per_site=MAX_KINASES_PER_SITE,)

for key, value in edge_stats.items():
    print(f"{key:<34} {value}")

kinsub_rows                        16945587
kinsub_kinases                     504
rows_targeting_our_sites           1915799
rows_after_resource_filter         123013
unique_edges_before_topk           60888
unique_edges                       60888
edges_by_class                     {'strict': 59249, 'literature': 1639}
sites_with_a_kinase                7616
kinases_used                       448
edges_per_site                     7.99


## 8. The sign rule, then the graph

A site may pass activation on to its own kinase only when its measured direction and its
PhosphoSitePlus `ON_FUNCTION` annotation agree that this means *more* activity (§4.1). Version 1
propagated from any site on a kinase protein, so known inhibitory sites switched their kinase on.

In [11]:
with KINSUB_PKL.open("rb") as handle:
    kinase_ids = set(pickle.load(handle)["source"].astype(str).unique())

sites = add_propagation_rule(site_table, kinase_ids)
print(sites["sign_evidence"].value_counts().to_string())
print()

candidate_graph, graph_stats = build_candidate_graph(sites,
                                                     edges,
                                                     root_kinase=EGFR_UNIPROT,)
for key, value in graph_stats.items():
    print(f"{key:<30} {value}")

sign_evidence
none         14346
induced        261
inhibited      157
ambiguous       33



nodes                          15246
edges                          61660
kinases                        448
phosphosites                   14797
sites_that_can_propagate       771
sites_blocked_by_sign_rule     33
sites_with_tf_bonus            0
reachable_from_root            4854
unreachable_from_root          10392


### 8.1 Why `MAX_KINASES_PER_SITE` is off

Keeping only the *k* highest-scoring kinases per site looks like the obvious way to cut the
prediction fan-out (§4.0), and it is wrong here. The rank is computed **per site**, and the root's
own edges rarely win it: motif scores rank tyrosine kinases almost identically, so EGFR is not in
the top 5 for most of its real substrates, and the cap severs the receptor from the graph. Measured
below: the resource filter costs little reachability, the rank cap costs nearly all of it.

Cut edges by stringency (resource, or a score threshold) — never by rank.

In [12]:
reachability = []
for label, regex, cap in [("Strict|literature + top5", "Strict|literature", 5,),
                          ("all resources + top5", None, 5,),
                          ("Strict|literature, no cap", "Strict|literature", None,),
                          ("all resources, no cap", None, None,)]:
    trial_edges, _ = load_kinase_substrate_edges(KINSUB_PKL,
                                                 site_ids=set(site_table["kinsub_site_id"]),
                                                 resource_regex=regex,
                                                 max_kinases_per_site=cap,)
    trial_graph, trial_stats = build_candidate_graph(sites,
                                                     trial_edges,
                                                     root_kinase=EGFR_UNIPROT,)
    reached = nx.descendants(trial_graph, "ROOT")
    reachability.append({"setting": label,
                         "edges": trial_stats["edges"],
                         "reachable": len(reached),
                         "kinases_reached": sum(1 for node in reached
                                                if trial_graph.nodes[node].get("node_type") == "kinase"),})

print(pd.DataFrame(reachability).to_string(index=False))

                  setting  edges  reachable  kinases_reached
 Strict|literature + top5  26021         11                1
     all resources + top5  65073         40                4
Strict|literature, no cap  61660       4853              130
    all resources, no cap 571400      13129              204


### The finding of step 2: the signal cannot leave the receptor

`reachable_from_root` is the number to look at. Breadth-first layers from `ROOT` show where
propagation stops and why.

In [13]:
labels = {node: (data.get("protein_site_id") or node) for node, data in candidate_graph.nodes(data=True)}

for depth, layer in enumerate(nx.bfs_layers(candidate_graph, "ROOT")):
    shown = [labels[node] for node in layer][:12]
    print(f"layer {depth}: {len(layer):>5} nodes   {shown}")

propagating = sites[sites["can_propagate"]]
reachable = nx.descendants(candidate_graph, "ROOT")
print()
print("sites sitting on a kinase protein (could continue the signal):", len(propagating))
print("of those, reachable from EGFR:", sum(1 for s in propagating["kinsub_site_id"] if s in reachable))

layer 0:     1 nodes   ['ROOT']
layer 1:     1 nodes   ['P00533']
layer 2:    28 nodes   ['LPXN_Y22', 'CTNND1_Y228', 'STAM2_Y192', 'EGFR_Y1110', 'EGFR_Y1172', 'EGFR_Y1197', 'EGFR_Y869', 'LDLR_Y828', 'LYN_Y397', 'CALM1_Y100', 'SRC_Y419', 'PKM_Y148']
layer 3:     4 nodes   ['P07948', 'P12931', 'P29317', 'Q05397']
layer 4:    43 nodes   ['PIK3R2_Y464', 'PPP1R12A_Y766', 'CDK1_Y15', 'SNRNP70_Y126', 'RBM5_Y57', 'CDK5_Y15', 'RIN1_Y36', 'SPRY2_Y55', 'CTNND1_Y280', 'MPZL1_Y263', 'TFRC_Y20', 'ERBB2_Y877']
layer 5:     5 nodes   ['Q00535', 'P04626', 'Q8N4C8', 'P00519', 'Q14289']
layer 6:   467 nodes   ['SH3PXD2B_S291', 'ZSWIM8_S1130', 'CBX4_S291', 'DNM1L_S616', 'EI24_S330', 'KMT2D_T2233', 'RIOK3_T150', 'SYNJ2_S1443', 'TNRC18_S416', 'TNRC18_S995', 'SIPA1L1_S162', 'IRAK2_S144']
layer 7:    25 nodes   ['O14730', 'O43187', 'O75385', 'O96013', 'P11274', 'P15056', 'P30291', 'Q14680', 'Q6P5Z2', 'Q7KZI7', 'Q96PY6', 'Q9UKI8']
layer 8:  1569 nodes   ['ATP1A1_S16', 'SIPA1L1_S1588', 'TPD52L2_S96', 'EPB41L2_S

## 9. Solve

In [14]:
start = time.perf_counter()
result = select_network(candidate_graph,
                        node_penalty=NODE_COST,
                        mip_gap=MIP_GAP,
                        time_limit=TIME_LIMIT_S,)
elapsed = time.perf_counter() - start

selected, n_feedback = tag_feedback_edges(result["subgraph"])

print(f"solver {result['solver']}, status {result['status']}, {elapsed:.1f} s")
print(f"candidate: {result['candidate_nodes']} nodes, {result['candidate_edges']} edges "
      f"({result['edges_dropped_by_time']} edges dropped as backwards in time)")
print(f"selected:  {selected.number_of_nodes()} nodes, {selected.number_of_edges()} edges, "
      f"{n_feedback} of them feedback")
print()
for key, value in result["decomposition"].items():
    print(f"{key:<32} {value:,.3f}")

solver SCIPY, status optimal, 23.7 s
candidate: 15246 nodes, 61649 edges (11 edges dropped as backwards in time)
selected:  48 nodes, 47 edges, 0 of them feedback

total_prize_on_offer             8,376.377
collected_prize                  67.730
missed_prize                     8,308.647
edge_cost                        23.585
node_cost                        14.100
latent_time_cost                 0.000
fraction_of_prize_collected      0.008


### The objective decomposition

This table is what exposed the central defect of version 1, where ~98% of the objective was a
constant no solution could reduce, so `NODE_COST` and the edge costs were doing all the work
(tutorial section 12.1). It must be printed on every run.

In [15]:
print(validate_selection(result).to_string(index=False))
print()

node_table, edge_table = selection_tables(result)
for row in edge_table.itertuples(index=False):
    print(f"  {row.source_label:<16} -> {row.target_label:<16} {row.edge_type:<22} w={row.weight:.2f}")

                        check   value  passed
                solver status optimal    True
                      acyclic    True    True
  nodes unreachable from root       0    True
          temporal violations       0    True
leaf nodes (impossible in v1)      38    True

  ROOT             -> P00533           root_to_kinase         w=1.00
  P00533           -> CTNND1_Y228      kinase_to_phosphosite  w=0.02
  P00533           -> STAM2_Y192       kinase_to_phosphosite  w=0.02
  P00533           -> EGFR_Y1110       kinase_to_phosphosite  w=0.07
  P00533           -> EGFR_Y1172       kinase_to_phosphosite  w=0.07
  P00533           -> EGFR_Y1197       kinase_to_phosphosite  w=0.14
  P00533           -> EGFR_Y869        kinase_to_phosphosite  w=0.06
  P00533           -> LYN_Y397         kinase_to_phosphosite  w=0.87
  P00533           -> CALM1_Y100       kinase_to_phosphosite  w=0.06
  P00533           -> PKM_Y148         kinase_to_phosphosite  w=0.00
  P00533           -> PLCG1_Y1253

### 9.1 Re-tuning `NODE_COST`

`NODE_COST = 1.2` came from version 1, where the sites were the top 10% by amplitude and the mean
prize was ~3.5. Here the prize is the fold change of every reproducibly responding site, median
**0.42**, so 1.2 charges more per node than most sites are worth and the network collapses onto the
receptor. The scan below is the re-tune §4.1 asks for.

In [16]:
scan = []
for cost in (0.05, 0.1, 0.2, 0.3, 0.5, 0.8, 1.2,):
    trial = select_network(candidate_graph,
                           node_penalty=cost,
                           mip_gap=MIP_GAP,
                           time_limit=TIME_LIMIT_S,)
    trial_graph, trial_feedback = tag_feedback_edges(trial["subgraph"])
    scan.append({"node_cost": cost,
                 "status": trial["status"],
                 "acyclic": nx.is_directed_acyclic_graph(trial_graph),
                 "nodes": trial_graph.number_of_nodes(),
                 "kinases": sum(1 for node in trial_graph
                                if trial_graph.nodes[node].get("node_type") == "kinase"),
                 "edges": trial_graph.number_of_edges(),
                 "leaves": sum(1 for node in trial_graph if trial_graph.out_degree(node) == 0),
                 "feedback": trial_feedback,
                 "prize_%": round(100 * trial["decomposition"]["fraction_of_prize_collected"], 2),})

print(pd.DataFrame(scan).to_string(index=False))

 node_cost  status  acyclic  nodes  kinases  edges  leaves  feedback  prize_%
      0.05 optimal     True     62        7     61      48         0     0.87
      0.10 optimal     True     57        6     56      45         0     0.85
      0.20 optimal     True     56        6     55      44         0     0.85
      0.30 optimal     True     48        5     47      38         0     0.81
      0.50 optimal     True     31        4     30      23         0     0.71
      0.80 optimal     True     19        1     18      17         0     0.61
      1.20 optimal     True     16        1     15      14         0     0.57


## 10. Observations, step 2

**A rank cap on kinases per site is the wrong knob, and it cost two hours of confusion.** With
`MAX_KINASES_PER_SITE = 5` the graph reached **11 nodes**; with the cap removed and the same
resource filter it reaches **4 854**. The rank is per site, and EGFR does not win the top-5 of its
own substrates because motif scores rank tyrosine kinases nearly identically. Section 8.1 keeps the
measurement.

**Even uncapped, the network stays receptor-local.** At `NODE_COST = 0.3` the selection is ~38
nodes with 4 kinases — EGFR, LYN, FAK/PTK2, CDK5 — and collects under 1% of the prize. ERK, RSK and
p38 never enter. The reason is visible in step 1: a kinase can only be entered through a **measured**
site on it that a **reachable** kinase phosphorylates, and MEK1's activation sites S218/S222 were
never measured in this dataset. The EGFR → RAS → RAF → MEK → ERK chain is broken at two places at
once — the non-kinase steps, and a missing measurement.

**This is not the PPI table's fault: the PPI layer is not in the model yet.** Everything above uses
the kinase-substrate table alone.

**Autophosphorylation forced an explicit constraint.** EGFR → EGFR_Y1172 → EGFR is a legal
two-cycle once equal activation times are allowed, and it costs only the epsilon edge charge, which
is far inside the MIP gap — so the solver returned it and the selection was not a DAG.
`select_network` now forbids `y(u,v) + y(v,u) <= 1`. Autophosphorylation is real, and it belongs in
the drawing as a feedback annotation (§8), not as a cycle in the selected graph.

Consequences for the next steps, now empirically justified rather than assumed:

- **Step 3 (orphan kinases, §4.2)** lets ERK, RSK and p38 enter at cost ω, since nothing can reach
  them from the receptor.
- **Step 4 (the SIGNOR causal layer, §4.3)** repairs the break itself, and must be allowed to carry
  a curated kinase → kinase activation even when the intermediate site was not measured —
  otherwise MEK stays missing whatever else is added.

Until one of those is in, the selected network is a receptor-local snapshot and must not be read as
a signalling cascade.

In [17]:
prefix = str(OUTPUT_PREFIX) + f"_step2_cost{str(NODE_COST).replace('.', 'p')}"

for table, suffix in ((node_table, ".selected_nodes.tsv",), (edge_table, ".selected_edges.tsv",),):
    path = Path(prefix + suffix)
    if path.exists():
        print("Exists already, not overwritten:", path.name)
    else:
        table.to_csv(path, sep="\t", index=False)
        print("Written:", path.name, table.shape)

Exists already, not overwritten: v2_site_table_EGF_ffdr0p05_peak_fc_step2_cost0p3.selected_nodes.tsv
Exists already, not overwritten: v2_site_table_EGF_ffdr0p05_peak_fc_step2_cost0p3.selected_edges.tsv


# Step 3 — orphan kinases, the virtual root

§4.2. Version 1 seeded the network only at EGFR, so a kinase whose substrates clearly respond
could not appear unless the receptor could reach it. Step 2 showed this is not hypothetical: ERK,
RSK and p38 are unreachable from EGFR in this dataset, because the receptor's substrates are
adaptors and MEK's activation sites were never measured.

`add_orphan_root_edges` adds an edge from the artificial root to **every** kinase, charged ω. The
selection becomes a forest — the EGF tree plus one small tree per orphan kinase — while the
formulation stays a single rooted flow problem.

ω is the parameter that decides what this model is:

- **small ω** degenerates the result into a kinase-substrate lookup, where every responding site is
  simply paired with its best-scoring kinase and nothing is explained;
- **large ω** reproduces step 2's receptor-only behaviour.

So it is scanned, not chosen.

Four of the six settings stop at the time limit rather than proving optimality, so read the
trend, not the individual rows.


In [18]:
# The scan is a script, not a cell: six large MILPs take ~20 minutes, and running them inside the
# notebook left the solver degraded for every cell after it — a selection that solves in 37 s
# standalone came back 'optimal_inaccurate' in 7 s. Heavy scans write tables; the notebook reads them.
#
#     python notebooks/07_dynamics_RPCST/run_step3_omega_scan.py

omega_scan = pd.read_csv(RESULTS_DIR / "v2_step3_omega_scan.tsv", sep="\t",)
print(omega_scan.to_string(index=False))

 omega  node_cost  secs             status  acyclic  nodes  kinases  orphan_roots  feedback  prize_pct                  markers
   0.5        0.3    44            optimal     True   3148      159           117         0       33.6 ERK2,AKT1,RSK1,p38a,JNK1
   1.0        0.3    34            optimal     True   3027      137            93         0       32.7 ERK2,AKT1,RSK1,p38a,JNK1
   2.0        0.3   333 optimal_inaccurate     True   2994      141            77         1       32.3 ERK1,ERK2,AKT1,RSK1,p38a
   4.0        0.3   621 optimal_inaccurate     True   2802      145            76         0       30.4 ERK1,ERK2,AKT1,RSK1,p38a
   8.0        0.3   522 optimal_inaccurate     True   2531      105            65         0       27.9 ERK1,ERK2,AKT1,RSK1,p38a
  16.0        0.3   313 optimal_inaccurate     True   2163       95            47         1       24.1 ERK1,ERK2,AKT1,RSK1,p38a


### 11.1 The (NODE_COST, omega) scan

The one-dimensional omega scan above answers the wrong question. omega is charged **once per orphan
root** (~50-110 of them); `NODE_COST` is charged **once per selected node** (~3000). The node cost
dominates, so the two have to be scanned together.

The scan is a standalone script rather than a cell, because it runs for ~40 minutes and writes its
results after every solve, so a run can be interrupted without losing completed rows:

```
python notebooks/07_dynamics_RPCST/run_step3_2d_scan.py
```

In [19]:
scan_2d = pd.read_csv(RESULTS_DIR / "v2_step3_2d_scan.tsv", sep="\t",)
print(scan_2d.to_string(index=False))

 node_cost  omega  secs             status  acyclic  cycle_rounds  cycles_removed  nodes  kinases  orphan_roots  edges  feedback  prize_pct                  markers
       0.5    1.0    25            optimal     True             0               0   1683       97            68   1690         0       23.5      ERK2,AKT1,RSK1,p38a
       0.5    4.0   559 optimal_inaccurate     True             1               1   1553      105            53   1597         2       21.5 ERK1,ERK2,AKT1,RSK1,p38a
       0.5   16.0   929 optimal_inaccurate     True             2               2   1376       99            42   1423         1       18.8 ERK1,ERK2,AKT1,RSK1,p38a
       1.0    1.0   197            optimal     True             0               0    598       44            27    597         0       12.0           ERK1,ERK2,AKT1
       1.0    4.0   354 optimal_inaccurate     True             0               0    553       45            23    674         0       10.9 ERK1,ERK2,AKT1,RSK1,p38a
       1.0

## 12. Observations, step 3

**Orphan entry does what it was added for.** All five marker kinases — ERK1, ERK2, AKT1, RSK1,
p38α — now enter somewhere, none of which was reachable from EGFR in step 2, and the first
**feedback edges** appear (1-3 at `NODE_COST = 0.5`). The mechanism from the first point of the
original notes is working: a kinase phosphorylating a site on a protein upstream of itself, as a
forward-in-time edge, with no cycle involved.

**omega is the second-order parameter.** Node cost 0.5 → 2.0 shrinks the network **23-fold**
(1683 → 72 nodes); a 16-fold omega increase shrinks it ~20%. The one-dimensional omega scan was
tuning the wrong knob.

**Cycle elimination holds.** All nine cells are acyclic. It fired only at `NODE_COST = 0.5`
(1-2 rounds) — exactly where large, loose networks make an epsilon-priced cycle affordable inside
the MIP gap.

**Solve time follows size, not omega.** The `NODE_COST = 2.0` cells prove optimality in ~36 s; the
`0.5` cells run 10-15 minutes and stop at the time limit. Four of nine cells are
`optimal_inaccurate` — good solutions, not proven best ones, and not quotable individually.

### ⚠️ The open problem this exposes

Collected prize falls from 23.5% to **1.9%** across the table. The only drawable network
(`NODE_COST = 2.0`, omega = 4: 59 nodes, 5 kinases, proven optimal in 37 s) explains **1.9% of the
reproducible EGF response**. Readability was bought by discarding almost all of the data, and
nothing in the objective resolves that trade-off.

The better route is the **prize floor** (§4.0 of the spec, and the "mixture of both" idea in the
original notes): filter first to the sites worth explaining — |log2FC| ≥ 0.5 keeps 5461 sites,
≥ 1.0 keeps 1602 — then use a *lower* node cost so the network explains most of *that* set.
"90% of the strong responders" is a defensible claim; "1.9% of everything" is not.

**One instability to measure, not to interpret:** ERK1 drops out at `NODE_COST = 2.0` while ERK2
stays, and JNK1 appeared only at low omega. ERK1 and ERK2 are near-identical substrates competing
for the same explanation, so which one survives is close to arbitrary. That is what the ensemble
check of §7 is for, and it should be run before any network here is read as biology.

### 12.1 The prize floor — controlling the model from the other side

Step 3 left a bad trade: a readable network cost 98% of the data. The prize floor attacks it from
the other end. Rather than charging more per node until the network is small, filter first to the
sites worth explaining, then use a *lower* node cost so the model explains most of **that** set.

```
python notebooks/07_dynamics_RPCST/run_step3_prize_floor_scan.py
```

`prize_pct` is relative to the floored set, which is the point: the claim under test is "most of
the strong responders", not "a fraction of everything".

In [20]:
floor_scan = pd.read_csv(RESULTS_DIR / "v2_step3_prize_floor_scan.tsv", sep="\t",)
print(floor_scan.to_string(index=False))

 prize_floor  sites_in  node_cost  omega  secs  status  acyclic  cycle_rounds  nodes  kinases  orphan_roots  edges  feedback  prize_pct                  markers
         0.5      5461        0.1    4.0    53 optimal     True             0   2777      127            58   2776         0       50.6 ERK1,ERK2,AKT1,RSK1,p38a
         0.5      5461        0.2    4.0    91 optimal     True             0   2646      111            46   2645         0       48.9 ERK1,ERK2,AKT1,RSK1,p38a
         0.5      5461        0.3    4.0    59 optimal     True             0   2496      100            39   2495         0       47.0 ERK1,ERK2,AKT1,RSK1,p38a
         0.5      5461        0.5    4.0    85 optimal     True             0   1503       72            27   1502         0       34.9 ERK1,ERK2,AKT1,RSK1,p38a
         1.0      1602        0.1    4.0     5 optimal     True             0    897       71            44    896         0       51.3      ERK1,ERK2,RSK1,p38a
         1.0      1602        0.2 

**Result.** Coverage improves 25-fold for the same readability — 1.9% of 14 797 sites at
`NODE_COST = 2.0` with no floor, against 46.7% of the 1 602 strong responders at floor 1.0. Every
cell is proven optimal, most in 5-10 s, where four of nine unfloored cells hit the time limit. And
`NODE_COST` stops mattering: a 5× increase moves the network 897 → 795 nodes, against a 23-fold
swing without a floor.

⚠️ **AKT1 is present at floor 0.5 and absent at floor 1.0.** Its sites respond more modestly, so the
stricter floor removes them. The floor is a biological choice, not a performance knob.

**Working default: floor 0.5, `NODE_COST` 0.3, ω 4** — 2496 nodes, 47% coverage, all five markers,
proven optimal in 59 s.

# Step 4 — the SIGNOR causal layer

§4.3, and the repair for what step 2 measured: with kinase → site → kinase edges only, the EGF
signal cannot leave the receptor, because EGFR's substrates are adaptors rather than kinases.

SIGNOR is the first resource in this project that can express a connection which is **not** a
phosphorylation. Two things are added (`add_signor_layer`):

1. **curated kinase → site edges**, where SIGNOR records a phosphorylation on a residue we
   measured — experimental evidence rather than motif prediction, and an independent check on the
   predicted table;
2. **causal protein → protein edges**, including non-kinase proteins, which — crucially — do
   **not** require the intermediate site to have been measured. Step 1 found MEK1's activation
   sites absent from this dataset, so a model that can only step through measured sites could never
   rebuild RAF → MEK → ERK however good the interaction data were.

The two exports on disk are **complementary**: `EGFR_16_09_26` carries the receptor-proximal layer,
`SIGNOR-EGF_14_08_26` carries GRB2 → SOS1 → HRAS → BRAF. `load_signor` takes a list.

In [21]:
from src.dynamic_network_v2 import (
    add_signor_layer,
    filter_by_prize_floor,
    augmented_edges,
    ensemble_selection,
    load_signor,
    readable_subnetwork,
    signor_gene_names,
    write_protein_centred_dot,
)

SIGNOR_FILES = [REPO_ROOT / "External_Data/Metadata/Signor/EGFR_16_09_26.tsv",
                REPO_ROOT / "External_Data/Metadata/Signor/SIGNOR-EGF_14_08_26.tsv",]
PRIZE_FLOOR = 0.5
OMEGA = 4.0

signor, signor_stats = load_signor(SIGNOR_FILES,)
for key, value in signor_stats.items():
    print(f"{key:<32} {value}")
MARKER_KINASES = {"P27361": "ERK1",
                  "P28482": "ERK2",
                  "P31749": "AKT1",
                  "Q15418": "RSK1",
                  "Q16539": "p38a",
                  "P45983": "JNK1",}


files                            ['EGFR_16_09_26.tsv', 'SIGNOR-EGF_14_08_26.tsv']
rows                             409
protein_to_protein               323
direct                           306
after_self_loops_dropped         293
after_slow_mechanisms_dropped    290
unique_edges                     167
proteins                         140
with_residue                     99
by_mechanism                     {'phosphorylation': 102, 'binding': 32, 'dephosphorylation': 15, 'polyubiquitination': 7, 'relocalization': 5, 'ubiquitination': 2, 'deubiquitination': 1, 'destabilization': 1, 'glycosylation': 1, 'guanine nucleotide exchange factor': 1}
by_effect                        {'up': 104, 'down': 56, 'unknown': 7}


In [22]:
floored = filter_by_prize_floor(site_table, min_abs_log2_fc=PRIZE_FLOOR,)
floored_sites = add_propagation_rule(floored, kinase_ids,)
floored_edges, _ = load_kinase_substrate_edges(KINSUB_PKL,
                                               site_ids=set(floored["kinsub_site_id"]),
                                               resource_regex=RESOURCE_REGEX,
                                               max_kinases_per_site=MAX_KINASES_PER_SITE,)
signor_graph, signor_graph_stats = build_candidate_graph(floored_sites,
                                                         floored_edges,
                                                         root_kinase=EGFR_UNIPROT,)
print("before SIGNOR: reachable from EGFR =", signor_graph_stats["reachable_from_root"])

layer_stats = add_signor_layer(signor_graph, signor, floored_sites, kinase_ids,)
print("after  SIGNOR: reachable from EGFR =", len(nx.descendants(signor_graph, "ROOT")))
print()
for key, value in layer_stats.items():
    print(f"{key:<32} {value}")

before SIGNOR: reachable from EGFR = 41
after  SIGNOR: reachable from EGFR = 1911

curated_site_edges_added         6
curated_edges_already_known      14
causal_edges_added               161
protein_nodes_added              110
site_edges_to_new_proteins       0
nodes                            6015
edges                            28743


In [23]:
add_orphan_root_edges(signor_graph, orphan_cost=OMEGA, root_kinase=EGFR_UNIPROT,)
signor_result = select_network(signor_graph,
                               node_penalty=NODE_COST,
                               mip_gap=MIP_GAP,
                               time_limit=TIME_LIMIT_S,)
signor_selection, signor_feedback = tag_feedback_edges(signor_result["subgraph"],)

print(f"status {signor_result['status']} | {signor_selection.number_of_nodes()} nodes, "
      f"{signor_selection.number_of_edges()} edges, {signor_feedback} feedback")
print(f"prize {100 * signor_result['decomposition']['fraction_of_prize_collected']:.1f}%")
print("node types:", collections.Counter(signor_selection.nodes[n].get("node_type") for n in signor_selection))
print("edge types:", collections.Counter(d.get("edge_type") for _, _, d in signor_selection.edges(data=True)))
print()
print("how each marker kinase was entered, and at what time:")
for accession, name in MARKER_KINASES.items():
    if accession in signor_selection:
        entered = [signor_selection.nodes[u].get("protein_site_id") or u
                   for u, _ in signor_selection.in_edges(accession)]
        print(f"   {name:<5} t={signor_selection.nodes[accession].get('assigned_activation_time')!s:<5} via {entered[:2]}")

status optimal | 2505 nodes, 2504 edges, 1 feedback
prize 47.0%
node types: Counter({'phosphosite': 2398, 'kinase': 104, 'protein': 2, 'root': 1})
edge types: Counter({'kinase_to_phosphosite': 2398, 'phosphosite_to_kinase': 60, 'orphan_to_kinase': 34, 'causal': 11, 'root_to_kinase': 1})

how each marker kinase was entered, and at what time:
   ERK1  t=5.0   via ['MAPK3_Y204']
   ERK2  t=0.0   via ['ROOT']
   AKT1  t=10.0  via ['AKT1_T479']
   RSK1  t=10.0  via ['RPS6KA1_S369']
   p38a  t=10.0  via ['MAPK14_T180']
   JNK1  t=5.0   via ['P45985']


## 13. Observations, step 4

**The layer does what it was added for: reachability from EGFR goes 41 → 1 911 nodes**, and the
selection contains a path that was structurally impossible in every earlier version:

```
ROOT -> EGFR -> PTPN11 -> SRC -> HRAS -> BRAF
        causal  causal    causal  causal
        phospho dephospho phospho binding
```

Three of those four edges are non-phosphorylation or non-kinase steps. At the working defaults the
selection is 2 505 nodes / 2 506 edges, 47.0% of the prize, proven optimal in 57 s, with 2 398
phosphosites, 104 kinases and 2 non-kinase proteins; 11 causal edges are used. Of the curated
SIGNOR kinase-site edges, **14 were already in the predicted table and 6 were new** — a modest but
real agreement check on the motif predictions.

### ⚠️ Two dead ends that only the full SIGNOR dump can fix

Neither export contains **BRAF → MEK1** or **MEK1 → ERK**, and MEK1's activation sites S218/S222
were never measured in this dataset. So the classic RAF → MEK → ERK link cannot be built from what
is on disk **by any route**, and MEK1 stays unreachable. This is the concrete argument for
downloading the full SIGNOR human dump — the pathway-scoped exports each cover a slice.

### ⚠️ ERK2 enters as an orphan root, not through the cascade

ERK2 is reachable at 7 hops once the causal layer is in, but the optimiser still pays ω = 4 to
enter it directly. The cascade exists in the candidate graph without being *chosen*. That is a
parameter question rather than a structural one, and it is examined in the next section.

## 14. The timing defect, and the fix — `TIME_DEFINITION = "onset"`

Step 4 left ERK2 entering through an orphan edge at t = 0 rather than through its own activation
site. That turned out to be the most informative failure in the whole build, because the cause is
not the cost model but **the time definition**.

A kinase's own activation-loop site **peaks after the kinase became active** — phosphorylation
accumulates while the kinase is already working. ERK2's Y187 and T185 both peak at 10 min, so
entering ERK2 through its own site forces `t(ERK2) >= 10` under the temporal constraint, and ERK2
can then no longer explain the 23 substrates it has at 2 and 5 min. Paying ω bought a free
activation time instead. The orphan edge was an escape hatch from a broken clock.

`onset` times a site by when its response **starts**: the first moment it reaches 50% of its own
peak, interpolated on the `log10(t + 1)` axis, anchored at the starve control and read in the
direction of the site's own response, so falling sites are timed like rising ones.

Two implementation points:

- **The prize is now taken from the argmax of |log2FC| regardless of `activation_time`.** A site is
  *timed* by when it starts and *prized* by how far it eventually moves. Under `peak_fc` the two
  coincide by construction — verified identical on all 14 797 sites, so this is a no-op there.
- `onset_time_continuous` keeps the interpolated value; `activation_time` is snapped to the
  measured grid so site times and latent kinase times live on the same scale.

In [24]:
onset_table, onset_ledger = build_site_table(DATA_PATH,
                                            max_ffdr=MAX_FFDR,
                                            time_definition="onset",
                                            cell_line=CELL_LINE,
                                            condition=CONDITION,)

print("activation-loop test under onset:")
print(activation_loop_report(onset_table).to_string(index=False))
print()
comparison = (site_table[["kinsub_site_id", "activation_time"]]
              .merge(onset_table[["kinsub_site_id", "activation_time", "onset_time_continuous"]],
                     on="kinsub_site_id",
                     suffixes=("_peak", "_onset",),))
earlier = comparison["activation_time_onset"].astype(float) < comparison["activation_time_peak"].astype(float)
print(f"sites timed earlier by onset: {earlier.sum()} ({100 * earlier.mean():.1f}%)")
print("peak-time distribution :", site_table["activation_time"].value_counts().sort_index().to_dict())
print("onset-time distribution:", onset_table["activation_time"].value_counts().sort_index().to_dict())

# Save it the same way as the peak_fc table: trimmed. Writing all 512 source columns produced a
# 114 MB file for 14797 sites.
onset_path = RESULTS_DIR / f"v2_site_table_{CONDITION}_ffdr{str(MAX_FFDR).replace('.', 'p')}_onset.tsv"
trim_site_table(onset_table,
                cell_line=CELL_LINE,
                condition=CONDITION,).to_csv(onset_path, sep="\t", index=False,)
print("written:", onset_path.name)


activation-loop test under onset:
protein_Id                      description      site_a      site_b time_a log2fc_a time_b log2fc_b  both_detected  times_agree
    P27361         ERK1 TEY activation loop P27361_T202 P27361_Y204      2    2.897      2    3.428           True         True
    P28482         ERK2 TEY activation loop P28482_T185 P28482_Y187      2    3.214      2    3.214           True         True
    Q02750             MEK1 activation loop Q02750_S218 Q02750_S222   <NA>     <NA>   <NA>     <NA>          False        False
    P36507             MEK2 activation loop P36507_S222 P36507_S226   <NA>     <NA>   <NA>     <NA>          False        False
    P31749 AKT1 activation, PDK1 and mTORC2 P31749_T308 P31749_S473   <NA>     <NA>   <NA>     <NA>          False        False

sites timed earlier by onset: 11670 (78.9%)
peak-time distribution : {'10': 2541, '15': 2743, '2': 476, '5': 8331, '90': 706}
onset-time distribution: {'10': 1318, '15': 920, '2': 8632, '5': 3926, 

written: v2_site_table_EGF_ffdr0p05_onset.tsv


### 14.1 Rebuilding on onset times

Everything from here on uses the onset table. This is also the pipeline in its final form: site
table → prize floor → kinase-substrate edges → candidate graph → SIGNOR causal layer → orphan
roots → ILP.

In [25]:
onset_floored = filter_by_prize_floor(onset_table, min_abs_log2_fc=PRIZE_FLOOR,)
onset_sites = add_propagation_rule(onset_floored, kinase_ids,)
onset_edges, _ = load_kinase_substrate_edges(KINSUB_PKL,
                                             site_ids=set(onset_floored["kinsub_site_id"]),
                                             resource_regex=RESOURCE_REGEX,
                                             max_kinases_per_site=MAX_KINASES_PER_SITE,)
network_graph, network_stats = build_candidate_graph(onset_sites,
                                                     onset_edges,
                                                     root_kinase=EGFR_UNIPROT,)
add_signor_layer(network_graph, signor, onset_sites, kinase_ids,)
add_orphan_root_edges(network_graph, orphan_cost=OMEGA, root_kinase=EGFR_UNIPROT,)

start = time.perf_counter()
onset_result = select_network(network_graph,
                              node_penalty=NODE_COST,
                              mip_gap=MIP_GAP,
                              time_limit=TIME_LIMIT_S,)
selected_network, onset_feedback = tag_feedback_edges(onset_result["subgraph"],)
print(f"status {onset_result['status']} in {time.perf_counter() - start:.0f} s | "
      f"{selected_network.number_of_nodes()} nodes, {onset_feedback} feedback edges, "
      f"prize {100 * onset_result['decomposition']['fraction_of_prize_collected']:.1f}%")
print()
print("marker kinases under onset timing:")
for accession, name in MARKER_KINASES.items():
    if accession in selected_network:
        entered = [selected_network.nodes[u].get("protein_site_id") or u
                   for u, _ in selected_network.in_edges(accession)]
        print(f"   {name:<5} t={selected_network.nodes[accession].get('assigned_activation_time')!s:<5} via {entered[:2]}")
print()
print(validate_selection(onset_result).to_string(index=False))

status optimal in 33 s | 2533 nodes, 2 feedback edges, prize 47.5%

marker kinases under onset timing:
   ERK1  t=2.0   via ['MAPK3_Y204']
   ERK2  t=2.0   via ['MAPK1_Y187']
   AKT1  t=2.0   via ['AKT1_S477']
   RSK1  t=2.0   via ['RPS6KA1_S380']
   p38a  t=2.0   via ['MAPK14_T180']
   JNK1  t=2.0   via ['P45985']

                        check   value  passed
                solver status optimal    True
                      acyclic    True    True
  nodes unreachable from root       0    True
          temporal violations       0    True
leaf nodes (impossible in v1)    2374    True


**Result.** Both detectable activation-loop pairs now agree (2 of 2, against 1 of 2 under
`peak_fc`): ERK1 T202/Y204 both at 2 min, ERK2 T185/Y187 both at 2 min. 78.9% of sites are timed
earlier and the t = 90 bin collapses from 706 sites to 1.

In the selection, every marker kinase moves from t = 5-10 to **t = 2**, ERK2 enters through
MAPK1_Y187 instead of the orphan edge, the solve takes 37 s instead of 98 s, and the collected
prize rises slightly (47.0% → 47.5%). `onset` is the module default from 2026-09-16.

# Step 5 — how much of this is real? (§7)

A single optimal network says what the cheapest explanation is, not how much better it is than the
next one. `ensemble_selection` re-solves under multiplicative noise on the edge costs and reports
how often each node and edge survives. `augmented_edges` lists the time-consistent candidate edges
between selected nodes that were *not* taken — the alternatives parsimony discarded.

In [26]:
ENSEMBLE_RUNS = 25
ENSEMBLE_NOISE = 0.2

edge_frequency, node_frequency, ensemble_runs = ensemble_selection(network_graph,
                                                                   node_penalty=NODE_COST,
                                                                   n_runs=ENSEMBLE_RUNS,
                                                                   noise=ENSEMBLE_NOISE,
                                                                   seed=1,
                                                                   mip_gap=MIP_GAP,
                                                                   time_limit=TIME_LIMIT_S,)

runs_table = pd.DataFrame(ensemble_runs)
print(f"{ENSEMBLE_RUNS} runs, all optimal: {(runs_table['status'] == 'optimal').all()}")
print(f"nodes per run: {runs_table['nodes'].min()} - {runs_table['nodes'].max()}")
print()
for label, table in (("edges", edge_frequency,), ("nodes", node_frequency,),):
    robust = (table["frequency"] >= 0.95).mean()
    middle = ((table["frequency"] >= 0.5) & (table["frequency"] < 0.95)).mean()
    coin = (table["frequency"] < 0.5).mean()
    print(f"{label:<6} n={len(table):>5}  >=95%: {robust:6.1%}   50-95%: {middle:6.1%}   <50%: {coin:6.1%}")

25 runs, all optimal: True
nodes per run: 2498 - 2556

edges  n= 8911  >=95%:   8.8%   50-95%:   8.5%   <50%:  82.7%
nodes  n= 2760  >=95%:  82.7%   50-95%:   9.3%   <50%:   8.0%


### ⚠️ The most important caveat in this notebook

**The node set is reproducible; the edges are not.** Measured on this run — 25 perturbed solves at
prize floor 0.5, all of which proved optimality, selecting 2505-2559 nodes each:

| | ≥95% of runs | 50-95% | <50% |
|---|---|---|---|
| **nodes** (n=2784) | **81.9%** | 9.0% | 9.1% |
| **edges** (n=8825) | 8.9% | 8.6% | **82.5%** |

Which sites belong in the EGF network is a result. **Which kinase explains a given site is mostly
arbitrary**, and that follows directly from §4.0: the kinase-substrate table offers ~43 candidate
kinases per site and motif scores cannot separate members of a family. Drawing a 12% edge with the
same arrow as a 100% edge would misrepresent the evidence, which is why the figure encodes
frequency as opacity.

⚠️ `ensemble_selection` **discards runs that did not prove optimality** rather than averaging them
in. This is not cosmetic: on an earlier notebook run where the solver degraded after ~40 minutes of
large MILPs, contaminated runs moved node stability from 82% to 68% and edge instability from 82%
to 86% — measuring the solver rather than the network.

In [27]:
augmented = augmented_edges(network_graph, selected_network,)
print(f"time-consistent candidate edges between selected nodes that were NOT taken: {len(augmented)}")
print(augmented.head(10).to_string(index=False))

time-consistent candidate edges between selected nodes that were NOT taken: 6345
source target source_label target_label        edge_type  cost
  ROOT Q9UEW8         ROOT       Q9UEW8 orphan_to_kinase   4.0
  ROOT P00519         ROOT       P00519 orphan_to_kinase   4.0
  ROOT Q7KZI7         ROOT       Q7KZI7 orphan_to_kinase   4.0
  ROOT Q9Y2H1         ROOT       Q9Y2H1 orphan_to_kinase   4.0
  ROOT P12931         ROOT       P12931 orphan_to_kinase   4.0
  ROOT Q9NRM7         ROOT       Q9NRM7 orphan_to_kinase   4.0
  ROOT O94921         ROOT       O94921 orphan_to_kinase   4.0
  ROOT O94804         ROOT       O94804 orphan_to_kinase   4.0
  ROOT Q13153         ROOT       Q13153 orphan_to_kinase   4.0
  ROOT Q13546         ROOT       Q13546 orphan_to_kinase   4.0


# Step 6 — the figure (§8)

The representation the user asked for: **each protein drawn surrounded by its own phosphosites**,
rather than version 1's layout where a kinase sat next to the sites it phosphorylates and its own
regulatory sites were scattered elsewhere. Every protein is a Graphviz cluster containing its
protein node and its measured sites, so a site that regulates a kinase sits *with* that kinase, and
the long arrows between clusters are the phosphorylation events.

Encoding:

| channel | meaning |
|---|---|
| node colour | activation time (onset), grey → blue → green → amber → red → purple |
| **solid fill** | phosphorylation **increases** |
| **hollow** (white fill, coloured border) | phosphorylation **decreases** |
| edge opacity | ensemble frequency — a coin-toss edge is drawn faint, with its % |
| dashed purple | SIGNOR causal (non-phosphorylation) step |
| dotted grey | site → its own protein (regulatory) |
| dashed black | orphan entry from the root |
| **red** | feedback: a kinase phosphorylating a site on a protein upstream of itself |

Hollow is used for down-regulation rather than a low alpha, because white label text on a pale fill
is unreadable — the user's request was alpha, and this keeps the intent (same colour, different
weight) while staying legible.

A 2 500-node selection is a result, not a picture, so `readable_subnetwork` keeps the paths from the
root to the cascade members and fills the remaining budget with the highest-prize sites hanging off
the proteins already included.

In [28]:
CASCADE_SEEDS = ["P00533",   # EGFR
                 "Q06124",   # PTPN11
                 "P12931",   # SRC
                 "P01112",   # HRAS
                 "P15056",   # BRAF
                 "P27361",   # ERK1
                 "P28482",   # ERK2
                 "Q15418",   # RSK1
                 "P31749",   # AKT1
                 "Q16539",]  # p38a
FIGURE_MAX_NODES = 90

gene_names = dict(zip(onset_sites["protein_Id"].astype(str),
                      onset_sites["protein_name"].astype(str),))
gene_names.update(signor_gene_names(signor,))

figure_graph = readable_subnetwork(selected_network,
                                   seeds=CASCADE_SEEDS,
                                   max_nodes=FIGURE_MAX_NODES,)
print(f"figure: {figure_graph.number_of_nodes()} nodes / {figure_graph.number_of_edges()} edges")

dot_path = write_protein_centred_dot(figure_graph,
                                     RESULTS_DIR / "v2_network_onset_floor0p5_cost0p3.dot",
                                     "EGF dynamic network v2 - onset timing, prize floor 0.5, node cost 0.3",
                                     gene_names=gene_names,)

if shutil.which("dot"):
    for fmt in ("svg", "png",):
        subprocess.run(["dot", f"-T{fmt}", "-o", str(dot_path.with_suffix("." + fmt)), str(dot_path)],
                       check=True,)
    print("rendered:", dot_path.with_suffix(".svg").name, "and", dot_path.with_suffix(".png").name)
else:
    print("Graphviz 'dot' not found; DOT written but not rendered")

figure: 90 nodes / 89 edges


rendered: v2_network_onset_floor0p5_cost0p3.svg and v2_network_onset_floor0p5_cost0p3.png


## 15. Observations, steps 5-6

**The figure reads as biology.** The SIGNOR causal chain EGFR → PTPN11 → SRC → HRAS → BRAF is
visible as dashed purple arrows, each protein carries its own phosphosites inside its own box, and
node colour shows that the whole receptor-proximal layer fires at 2 min.

**What may be claimed from this, and what may not.**

- *May*: which sites are part of the EGF response (81.9% of nodes are selected in ≥95% of perturbed
  runs), that the cascade is reconstructible once non-phosphorylation steps are allowed, and the
  activation times now that they pass the activation-loop test.
- *May not*: that a particular kinase phosphorylates a particular site. 82.5% of edges are chosen in
  under half of the perturbed runs. The arrows are hypotheses weighted by opacity, not assignments.

⚠️ **Feedback works, but individual feedback edges are unstable.** Repeated solves of the *same*
configuration gave 0, 1 and 2 tagged feedback edges — one solution had SRC phosphorylating EGFR
Y1110 and Y1172, another lost it entirely. The count for this run is printed by §14.1 above, and
the figure draws any feedback edge in magenta, labelled. So feedback is expressible, which was the
goal of the first point in the original notes, and the *mechanism* is a result; but no individual
feedback edge from a single solve is. It is the edge instability measured above, landing on exactly
the edges one would most want to interpret.

**Known limitations carried forward, in order of how much they matter:**

1. **MEK1 is unreachable and cannot be fixed from the files on disk** — neither SIGNOR export has
   BRAF → MEK1 or MEK1 → ERK, and MEK1 S218/S222 were never measured. The full SIGNOR human dump is
   the fix.
2. **The dataset predates the 2026-09-15 channel-offset correction**, so the prizes inherit the
   +0.566 median log2FC shift at 5 min. A re-run on a `chcorr` table is the sensitivity check.
3. **Solver drift is real.** The same configuration re-solved gives slightly different networks
   (2505-2559 nodes), and long notebook runs degrade HiGHS. Heavy scans therefore live in scripts
   that write tables, and the notebook reads them.
4. **`max_step` is still unimplemented**, so only two of the three time definitions of §3 exist.

# 16. Why ERK is not activated the canonical way — and why more SIGNOR will not fix it

The obvious reading of step 4 was that the cascade is incomplete because the SIGNOR exports on disk
are pathway-scoped slices: neither contains BRAF → MEK. That reading is **wrong**, and the cell
below is the test that settles it.

**First, a correction.** MEK → ERK was never missing. The kinase-substrate table carries
MAP2K2 → MAPK1_Y187 at score 0.85 and MAP2K1 → MAPK1_T185 at 0.90, and the selection *uses* them —
the route to ERK2 is:

```
ROOT -> Q86V86 (orphan) -> MAP2K2_S306 -> MAP2K2 -> MAPK1_Y187 -> ERK2
```

ERK2 **is** activated through MEK. What is not canonical is how MEK gets activated: through its own
site S306, phosphorylated by an orphan kinase, rather than by RAF coming from EGFR.

**The test.** Inject the curated RAF → MEK activation edges that the full SIGNOR human dump
contains and these exports lack, then re-solve. If the data were the limitation, the cascade should
now be chosen.

In [ ]:
# Counterfactual only — these three rows are NOT written to disk and are used by no saved result.
# They stand in for what the full SIGNOR human dump would supply.
counterfactual = pd.DataFrame([
    dict(source="P15056", target="Q02750", source_name="BRAF", target_name="MAP2K1",
         effect="up", mechanism="phosphorylation", residue="S218", score=0.9, cost=0.065,),
    dict(source="P15056", target="P36507", source_name="BRAF", target_name="MAP2K2",
         effect="up", mechanism="phosphorylation", residue="S222", score=0.9, cost=0.065,),
    dict(source="P04049", target="Q02750", source_name="RAF1", target_name="MAP2K1",
         effect="up", mechanism="phosphorylation", residue="S218", score=0.9, cost=0.065,),])

test_graph, _ = build_candidate_graph(onset_sites, onset_edges, root_kinase=EGFR_UNIPROT,)
add_signor_layer(test_graph,
                 pd.concat([signor, counterfactual], ignore_index=True,),
                 onset_sites,
                 kinase_ids,)
add_orphan_root_edges(test_graph, orphan_cost=OMEGA, root_kinase=EGFR_UNIPROT,)
test_result = select_network(test_graph,
                             node_penalty=NODE_COST,
                             mip_gap=MIP_GAP,
                             time_limit=TIME_LIMIT_S,)
test_selection, _ = tag_feedback_edges(test_result["subgraph"],)

label = lambda node: test_selection.nodes[node].get("protein_site_id") or gene_names.get(node, node)
print("cascade proteins present in the selection:",
      [gene_names.get(a, a) for a in ["P00533", "Q06124", "P12931", "P01112", "P15056", "P36507", "P28482"]
       if a in test_selection])
print("BRAF -> MEK2 in candidate graph:", test_graph.has_edge("P15056", "P36507"),
      "| selected:", test_selection.has_edge("P15056", "P36507"))
print("MEK2 is explained by:", [label(u) for u, _ in test_selection.in_edges("P36507")])
print("BRAF -> MEK2 listed as an available-but-unselected alternative:",
      len(augmented_edges(test_graph, test_selection,).query("source == 'P15056' and target == 'P36507'")) > 0)
print("Q86V86 explains", test_selection.out_degree("Q86V86"), "sites for a single omega payment")

## The diagnosis: the objective, not the data

Adding the curated edges changes **nothing** — the same path is chosen, and BRAF → MEK2 sits
unselected in the augmented list. Every cascade protein is already in the network: EGFR, PTPN11,
SRC, HRAS, BRAF, MAP2K2, MAPK1. The canonical route is not absent, it is **unpurchased**.

The reason is constraint **C2**: a node needs *at least one* incoming edge, every edge costs
something, and redundancy earns no prize. MEK2 already has an explanation through its own site, so
buying BRAF → MEK2 would add cost and collect nothing. **Parsimony buys exactly one explanation per
node, and the cheapest one wins** — it has no notion of which explanation is better established.

A second effect compounds it: **Q86V86 explains 46 sites** in this selection and has no path from
EGFR. ω is charged once per orphan *root*, so a hub that explains dozens of sites costs the same 4.0
as one explaining a single site. Cheap hubs outcompete real cascades.

## What I would fix, in order

1. **Reward curated evidence** (the real fix for this). Give SIGNOR and literature edges a bonus —
   a negative effective cost, or a prize for using one — so an established explanation can outbid a
   cheap predicted one. This states "prefer established biology" as a modelling choice instead of
   letting cost silently decide. It is the change most likely to make the cascade appear.
2. **Charge ω per explained site, not per orphan root**, or cap an orphan's out-degree. A kinase
   with no upstream connection explaining 46 sites should not be cheaper than a five-step cascade.
3. **Draw the augmented edges faintly.** BRAF → MEK2 is already computed; showing it alongside the
   selection is truthful about what the data cannot distinguish, and costs nothing.
4. **Re-run on a `chcorr` table** (§2.5) — the prizes currently inherit the +0.566 median log2FC
   shift at EGF 5 min from the missing loading normalisation.
5. **Implement `max_step`** (§3), the one time definition still unbuilt, and re-run the
   activation-loop test across all three.
6. **The full SIGNOR human dump** — still worth having for coverage, but now known *not* to be the
   fix for this. Expect it to add candidate routes, not to change which one is chosen.

Items 1 and 2 change what the model prefers and should be decided deliberately rather than tuned;
item 3 changes only the figure and could be done immediately.